In [1]:
import requests
import pandas as pd
import time

In [7]:
test_points = [
    ("Alba Iulia centru", 46.0733, 23.5800),
    ("Alba - sud",        45.90,   23.57),
    ("Alba - nord",       46.30,   23.57),
    ("Cluj centru",       46.7712, 23.6236),
    ("Bucuresti centru",  44.4268, 26.1025),
]

BASE_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

for nume, lat, lon in test_points:
    params = {
        "lon": lon, "lat": lat,
        "property": ["phh2o", "clay"],
        "depth": "0-5cm",
        "value": "mean"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()
    
    phh2o = data["properties"]["layers"][0]["depths"][0]["values"]["mean"]
    clay  = data["properties"]["layers"][1]["depths"][0]["values"]["mean"]
    
    print(f"{nume}: status={r.status_code} | phh2o={phh2o} | clay={clay}")

Alba Iulia centru: status=200 | phh2o=None | clay=None
Alba - sud: status=200 | phh2o=345 | clay=57
Alba - nord: status=200 | phh2o=311 | clay=54
Cluj centru: status=200 | phh2o=None | clay=None
Bucuresti centru: status=200 | phh2o=None | clay=None


In [10]:


judete = {
    "Alba":             (46.07, 23.57),
    "Arad":             (46.17, 21.32),
    "Arges":            (45.08, 24.87),
    "Bacau":            (46.57, 26.91),
    "Bihor":            (47.05, 21.95),
    "Bistrita-Nasaud":  (47.13, 24.50),
    "Botosani":         (47.75, 26.67),
    "Brasov":           (45.65, 25.61),
    "Braila":           (45.27, 27.96),
    "Buzau":            (45.15, 26.82),
    "Caras-Severin":    (45.30, 22.07),
    "Calarasi":         (44.20, 26.33),
    "Cluj":             (46.77, 23.59),
    "Constanta":        (44.18, 28.65),
    "Covasna":          (45.87, 26.18),
    "Dambovita":        (44.93, 25.45),
    "Dolj":             (44.31, 23.80),
    "Galati":           (45.43, 28.05),
    "Giurgiu":          (43.90, 25.97),
    "Gorj":             (44.95, 23.27),
    "Harghita":         (46.43, 25.57),
    "Hunedoara":        (45.75, 22.90),
    "Ialomita":         (44.60, 27.37),
    "Iasi":             (47.16, 27.59),
    "Ilfov":            (44.53, 26.17),
    "Maramures":        (47.66, 23.57),
    "Mehedinti":        (44.63, 22.65),
    "Mures":            (46.55, 24.56),
    "Neamt":            (46.97, 26.38),
    "Olt":              (44.43, 24.37),
    "Prahova":          (45.07, 25.99),
    "Satu Mare":        (47.80, 22.87),
    "Salaj":            (47.19, 23.06),
    "Sibiu":            (45.80, 24.15),
    "Suceava":          (47.65, 26.25),
    "Teleorman":        (43.98, 25.00),
    "Timis":            (45.75, 21.22),
    "Tulcea":           (45.18, 29.02),
    "Vaslui":           (46.64, 27.73),
    "Valcea":           (45.10, 24.37),
    "Vrancea":          (45.70, 27.18),
    "Bucuresti":        (44.43, 26.10),
}

PROPERTIES = ["phh2o", "clay", "sand", "silt", "soc", "nitrogen", "cec"]
DEPTHS = ["0-5cm", "5-15cm", "15-30cm"]
BASE_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

# 9 puncte per județ: centrul + 8 offset-uri de ~15km
OFFSETS = [
    (0, 0),
    (0.15, 0), (-0.15, 0), (0, 0.15), (0, -0.15),
    (0.15, 0.15), (-0.15, 0.15), (0.15, -0.15), (-0.15, -0.15),
]

def fetch_one_point(lat, lon):
    """Fetch date sol pentru un singur punct. Returneaza dict sau None."""
    params = [("lon", lon), ("lat", lat)]
    for p in PROPERTIES:
        params.append(("property", p))
    for d in DEPTHS:
        params.append(("depth", d))
    params.append(("value", "mean"))

    r = requests.get(BASE_URL, params=params, timeout=30)
    if r.status_code != 200:
        return None

    data = r.json()
    layers = data["properties"]["layers"]

    row = {}
    for layer in layers:
        name = layer["name"]
        d_factor = layer["unit_measure"]["d_factor"]
        vals = [
            depth["values"]["mean"] / d_factor
            for depth in layer["depths"]
            if depth["values"]["mean"] is not None
        ]
        row[name] = round(sum(vals) / len(vals), 3) if vals else None

    # considera valid doar daca cel putin 5 din 7 proprietati au date
    valid_count = sum(1 for v in row.values() if v is not None)
    return row if valid_count >= 5 else None

def fetch_soilgrids_judet(judet, lat, lon):
    """Incearca 9 puncte, face media celor valide."""
    all_rows = []

    for dlat, dlon in OFFSETS:
        result = fetch_one_point(lat + dlat, lon + dlon)
        if result:
            all_rows.append(result)
        time.sleep(0.5)

    if not all_rows:
        print(f"  [{judet}] ESUAT — niciun punct valid din {len(OFFSETS)}")
        return {"Judet": judet, "lat": lat, "lon": lon, **{p: None for p in PROPERTIES}}

    # media peste toate punctele valide
    row = {"Judet": judet, "lat": lat, "lon": lon, "puncte_valide": len(all_rows)}
    for prop in PROPERTIES:
        vals = [r[prop] for r in all_rows if r.get(prop) is not None]
        row[prop] = round(sum(vals) / len(vals), 3) if vals else None

    print(f"  [{judet}] OK — {len(all_rows)}/{len(OFFSETS)} puncte valide, phh2o={row.get('phh2o')}")
    return row

results = []
for i, (judet, (lat, lon)) in enumerate(judete.items(), 1):
    print(f"[{i}/{len(judete)}] {judet}...")
    row = fetch_soilgrids_judet(judet, lat, lon)
    results.append(row)
    time.sleep(1)

df = pd.DataFrame(results)
df.to_csv("sol_judete_romania.csv", index=False)
print(f"\nSalvat {len(df)} judete")
print(f"Complete: {df['phh2o'].notna().sum()}/{len(df)}")
print(df[["Judet", "phh2o", "soc", "puncte_valide"]].to_string())



[1/42] Alba...
  [Alba] OK — 6/9 puncte valide, phh2o=6.283
[2/42] Arad...
  [Arad] OK — 8/9 puncte valide, phh2o=6.612
[3/42] Arges...
  [Arges] OK — 8/9 puncte valide, phh2o=5.763
[4/42] Bacau...
  [Bacau] OK — 8/9 puncte valide, phh2o=6.604
[5/42] Bihor...
  [Bihor] OK — 8/9 puncte valide, phh2o=6.312
[6/42] Bistrita-Nasaud...
  [Bistrita-Nasaud] OK — 8/9 puncte valide, phh2o=6.104
[7/42] Botosani...
  [Botosani] OK — 6/9 puncte valide, phh2o=6.494
[8/42] Brasov...
  [Brasov] OK — 7/9 puncte valide, phh2o=6.0
[9/42] Braila...
  [Braila] OK — 7/9 puncte valide, phh2o=7.619
[10/42] Buzau...
  [Buzau] OK — 8/9 puncte valide, phh2o=7.013
[11/42] Caras-Severin...
  [Caras-Severin] OK — 9/9 puncte valide, phh2o=5.685
[12/42] Calarasi...
  [Calarasi] OK — 9/9 puncte valide, phh2o=6.733
[13/42] Cluj...
  [Cluj] OK — 8/9 puncte valide, phh2o=6.5
[14/42] Constanta...
  [Constanta] OK — 3/9 puncte valide, phh2o=7.578
[15/42] Covasna...
  [Covasna] OK — 8/9 puncte valide, phh2o=6.179
[16/42] Da